# Week 11: Data Acquisition

By the end of this worksheet you will be able to

- Understand the fundamentals of Application Programming Interfaces (APIs) and how they facilitate the access and retrieval of data.  
- Use APIs to acquire data from the web.  
- Perform web scraping to extract data from the web.  
- Implement computer code to read in multi-file data sets.  
- Describe ethical considerations when acquiring data from the web  


## Getting Started

Before beginning the worksheet, let us load the necessary packages we will be using through out the worksheet

In [ ]:
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(httr))
suppressPackageStartupMessages(library(jsonlite))
suppressPackageStartupMessages(library(rvest))
suppressPackageStartupMessages(library(arrow))

## Data Acquistion 
 Until this point, we were loading data locally using functions like `read_csv()`. In this worksheet, you will be collecting data from various sources on internet using the follwoing methods:
- **APIs (Application Programming Interfaces):** Allow programmatic access to data from web services in structured formats like JSON or XML.
- **Web Scraping:** Extracts data directly from web pages when APIs are unavailable.
- **Parquet Files:** A columnar storage file format optimized for large-scale data processing.

### PART 1: API

There are two parts to making an API request:
- The resource you want to request (i.e., the data)  
- The verb specifying what you want to do with that resource (get it, update it, create one etc.)

The **resource** can be accessed using a URL or web address.

The base URL is the domain and part of the path that is included on all resources. It acts as the “root” for any particular resource.
For example, the GitHub API has a base URL of https://api.github.com/.

An endpoint specifies which resource on that domain you want to access.
For example, a specific user or organization on GitHub such as https://api.github.com/users/katieburak.

We will be using the `httr` package in R, which has functions for the following most important http verbs:

`GET()`: Return a representation of the current state of the resource  
`HEAD()`: Return the header of the resource  
`PATCH()`: Update a portion of the resource’s state  
`PUT()`: Update the resource to have a new state  
`DELETE()`: Remove the resource  
`POST()`: Add a new subresource  

We will be using the **Frankfurter API** which is a free, open-source API that provides current and historical foreign exchange rates published by the European Central Bank. You can find more information about this API on their website (https://frankfurter.dev/). An example of extracting the current conversion rate of USD to EUR:

In [ ]:
url <- "https://api.frankfurter.app/latest?from=USD&to=EUR" # Resource URL
response <- GET(url) # Make a GET request to the resource
data <- response |> content(as = "text") |> fromJSON() # Parsing the JSON response
rates <- data |> as_tibble() |> pull(rates) |> as_tibble() |> pull(EUR) # Converting parsed data into a tibble

rates

Let's break down the URL from the above code chunk:

- `https://api.frankfurter.app/latest` is the base URL and endpoint for the latest rates.  
- `?from=USD&to=EUR` are query parameters specifying the base currency (USD) and the target currency (EUR).

Other query parameters include:

- `from` - Base currency (3-letter ISO code)	
- `to` - Target currency or list of currencies (comma-separated)	
- `amount` - Amount of the base currency to convert	
- `start_date..end_date` - Range for historical time-series data (used in place of `/latest` endpoint)	


As an example, let's change the endpoint from latest currency rates to all the available currencies

In [ ]:
url <- "https://api.frankfurter.app/currencies" # Resource URL with currencies as endpoint
response <- GET(url)
data <- response |> 
        content(as = "text") |> 
        fromJSON() |> 
        enframe(name = "ISO Code", value = "Currency") |> 
        unnest(cols = Currency)

head(data)

**Question 1** 

What were the exchange rates from CAD to EUR, GBP and USD on January 15, 2024 as a tibble? Assign the table to variable `answer1`(note that you are required to do this programatically rather than storing simply storing the values). HINT: Here is the start of the url for that particular date (`https://api.frankfurter.app/2024-01-15`), but you will have to change the url endpoint based on the exchnage rates you want, separating currencies with commas (e.g., `USD,EUR,GBP`).(1 point) 

In [ ]:

# answer1<- "FILL IN THE ANSWER HERE"

# YOUR CODE HERE
fail()

answer1

In [ ]:
library(digest)
stopifnot("answer1 should be a data frame"= setequal(digest(paste(toString('data.frame' %in% class(answer1)), "a3d26")), "cb1e59b7dde0a389ee6e51fa627d3974"))
stopifnot("dimensions of answer1 are not correct"= setequal(digest(paste(toString(dim(answer1)), "a3d26")), "11b2307bd03661dbd1f807ffb770fbaa"))
stopifnot("column names of answer1 are not correct"= setequal(digest(paste(toString(sort(colnames(answer1))), "a3d26")), "5468baad8254b1e9d79e4f103e062e77"))
stopifnot("types of columns in answer1 are not correct"= setequal(digest(paste(toString(sort(unlist(sapply(answer1, class)))), "a3d26")), "3bd822f65c6ad4a2860f99905485cd10"))
stopifnot("values in one or more numerical columns in answer1 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer1, is.numeric))) sort(round(sapply(answer1[, sapply(answer1, is.numeric)], sum, na.rm = TRUE), 2)) else 0), "a3d26")), "f24ad236e600b4b13d023ef9a28edadd"))
stopifnot("values in one or more character columns in answer1 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer1, is.character))) sum(sapply(answer1[sapply(answer1, is.character)], function(x) length(unique(x)))) else 0), "a3d26")), "c5469260fe6dbff5fbfc2c6d0aa9a8bc"))
stopifnot("values in one or more factor columns in answer1 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer1, is.factor))) sum(sapply(answer1[, sapply(answer1, is.factor)], function(col) length(unique(col)))) else 0), "a3d26")), "c5469260fe6dbff5fbfc2c6d0aa9a8bc"))

print('Success!')


Now let's retrieve data from January 2-31 2024 for USD to EUR conversion rates.

In [ ]:
start_date <- "2024-01-02"
end_date <- "2024-01-31"
url <- paste0("https://api.frankfurter.app/", start_date, "..", end_date, "?from=USD&to=EUR")
response <- GET(url)
data <- response |> content(as = "text") |> fromJSON()
rates_tibble <- data |> as_tibble() |> pull(rates) |> enframe(name = "date", value = "rates") |>  unnest_wider(rates)
colnames(rates_tibble) <- c("date", "exchange_rate")
rates_tibble <- rates_tibble|> mutate(date = as.Date(date))

In [ ]:
rates_tibble

**Question 2** 

Plot the trend of the above rates as a line plot. Assign the answer to variable `answer2`. (1 point)

In [ ]:
#answer2 <- "FILL IN THE ANSWER HERE"

# YOUR CODE HERE
fail()

answer2

In [ ]:
library(digest)
stopifnot("type of plot is not correct (if you are using two types of geoms, try flipping the order of the geom objects!)"= setequal(digest(paste(toString(sapply(seq_len(length(answer2$layers)), function(i) {c(class(answer2$layers[[i]]$geom))[1]})), "c7ecb")), "87940fe36d8e1ecfde0247d3ebb66783"))
stopifnot("variable x is not correct"= setequal(digest(paste(toString(unlist(lapply(sapply(seq_len(length(answer2$layers)), function(i) {rlang::get_expr(c(answer2$layers[[i]]$mapping, answer2$mapping)$x)}), as.character))), "c7ecb")), "cde30bbef9051e8f4e4408c48ce7bf2f"))
stopifnot("variable y is not correct"= setequal(digest(paste(toString(unlist(lapply(sapply(seq_len(length(answer2$layers)), function(i) {rlang::get_expr(c(answer2$layers[[i]]$mapping, answer2$mapping)$y)}), as.character))), "c7ecb")), "f38ff59731a923425ba0f1f7fcee7983"))
stopifnot("x-axis label is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer2$layers[[1]]$mapping, answer2$mapping)$x)!= answer2$labels$x), "c7ecb")), "0cc9b7a03dcf498c514c65b1c306361a"))
stopifnot("y-axis label is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer2$layers[[1]]$mapping, answer2$mapping)$y)!= answer2$labels$y), "c7ecb")), "0cc9b7a03dcf498c514c65b1c306361a"))
stopifnot("incorrect colour variable in answer2, specify a correct one if required"= setequal(digest(paste(toString(rlang::get_expr(c(answer2$layers[[1]]$mapping, answer2$mapping)$colour)), "c7ecb")), "1a0f432972805943ffd647c4cb855300"))
stopifnot("incorrect shape variable in answer2, specify a correct one if required"= setequal(digest(paste(toString(rlang::get_expr(c(answer2$layers[[1]]$mapping, answer2$mapping)$shape)), "c7ecb")), "1a0f432972805943ffd647c4cb855300"))
stopifnot("the colour label in answer2 is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer2$layers[[1]]$mapping, answer2$mapping)$colour) != answer2$labels$colour), "c7ecb")), "1a0f432972805943ffd647c4cb855300"))
stopifnot("the shape label in answer2 is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer2$layers[[1]]$mapping, answer2$mapping)$colour) != answer2$labels$shape), "c7ecb")), "1a0f432972805943ffd647c4cb855300"))
stopifnot("fill variable in answer2 is not correct"= setequal(digest(paste(toString(quo_name(answer2$mapping$fill)), "c7ecb")), "e4415ca1b42bcf7816371ad9eed5dd60"))
stopifnot("fill label in answer2 is not informative"= setequal(digest(paste(toString((quo_name(answer2$mapping$fill) != answer2$labels$fill)), "c7ecb")), "1a0f432972805943ffd647c4cb855300"))
stopifnot("position argument in answer2 is not correct"= setequal(digest(paste(toString(class(answer2$layers[[1]]$position)[1]), "c7ecb")), "f3e0da26f6df895f8fa8a1359f9296e3"))

print('Success!')

**Question 3**

Which of the following best describes the overall trend in the line plot? Assign your quoted answer to `answer3`. (1 point)
 
  A) Increasing  
  B) Decreasing  
  C) Stable  

In [ ]:
#answer3 <- "FILL IN THE ANSWER HERE" #Option A, B or C

# YOUR CODE HERE
fail()

In [ ]:
library(digest)
stopifnot("type of answer3 is not character"= setequal(digest(paste(toString(class(answer3)), "b071f")), "17da6196e451c5de690cda6b1e22f7af"))
stopifnot("length of answer3 is not correct"= setequal(digest(paste(toString(length(answer3)), "b071f")), "bb925d21ea71fe77b997d3df52d70f85"))
stopifnot("value of answer3 is not correct"= setequal(digest(paste(toString(tolower(answer3)), "b071f")), "562ee9fb90525a459c6fd61b05c6de93"))
stopifnot("letters in string value of answer3 are correct but case is not correct"= setequal(digest(paste(toString(answer3), "b071f")), "aed3fe97c45ddffb569b7d108744d4bd"))

print('Success!')

**Question 4**

Say you went on a work trip to France on "2024-06-12". You need to submit an expense report on how much you spent there in CAD for your reimbursments and all your reciepts show the spending in EUR to a total of 260 EUR. Use the Frankfurter API to  convert this EUR amount to CAD with the conversion rate at the day of spending. Assign the CAD value to the variable `answer4`. (1 point)

In [ ]:
#answer4 <- "FILL IN THE ANSWER HERE"

# YOUR CODE HERE
fail()

answer4

In [ ]:
library(digest)
stopifnot("type of answer4 is not numeric"= setequal(digest(paste(toString(class(answer4)), "ead8d")), "a40905c8afdec5b199f540a159656519"))
stopifnot("value of answer4 is not correct (rounded to 2 decimal places)"= setequal(digest(paste(toString(round(answer4, 2)), "ead8d")), "3c1e3fbab84460be3a1e4aa9816565db"))
stopifnot("length of answer4 is not correct"= setequal(digest(paste(toString(length(answer4)), "ead8d")), "916ab989a9a037636fe67657830556f5"))
stopifnot("values of answer4 are not correct"= setequal(digest(paste(toString(sort(round(answer4, 2))), "ead8d")), "3c1e3fbab84460be3a1e4aa9816565db"))

print('Success!')

**Question 5**

Analyze the volatility of the Canadian Dollar (CAD) against USD, EUR, and GBP during the COVID-19 pandemic (March 2020 to May 2023). Plot the trend of CAD conversion rates to the three currencies over the time period. Assign the ggplot object to `answer5`. HINT: you may need to pivot your data into long format. (1 point)

In [ ]:
# YOUR CODE HERE
fail()

answer5

In [ ]:
library(digest)
stopifnot("type of plot is not correct (if you are using two types of geoms, try flipping the order of the geom objects!)"= setequal(digest(paste(toString(sapply(seq_len(length(answer5$layers)), function(i) {c(class(answer5$layers[[i]]$geom))[1]})), "e6096")), "bbd4bbd965dd255a8e19a3f301affc11"))
stopifnot("variable x is not correct"= setequal(digest(paste(toString(unlist(lapply(sapply(seq_len(length(answer5$layers)), function(i) {rlang::get_expr(c(answer5$layers[[i]]$mapping, answer5$mapping)$x)}), as.character))), "e6096")), "06dfdce4e773fddcaffb1ff104dbabc0"))
stopifnot("variable y is not correct"= setequal(digest(paste(toString(unlist(lapply(sapply(seq_len(length(answer5$layers)), function(i) {rlang::get_expr(c(answer5$layers[[i]]$mapping, answer5$mapping)$y)}), as.character))), "e6096")), "6d5bd4614eed9e3c7acdbdb5e4b56feb"))
stopifnot("x-axis label is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer5$layers[[1]]$mapping, answer5$mapping)$x)!= answer5$labels$x), "e6096")), "746d1a67b388ecdb3b41e2991f6c7a9b"))
stopifnot("y-axis label is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer5$layers[[1]]$mapping, answer5$mapping)$y)!= answer5$labels$y), "e6096")), "746d1a67b388ecdb3b41e2991f6c7a9b"))
stopifnot("incorrect colour variable in answer5, specify a correct one if required"= setequal(digest(paste(toString(rlang::get_expr(c(answer5$layers[[1]]$mapping, answer5$mapping)$colour)), "e6096")), "234d84f694dba6ba8f7d6119067bf6c6"))
stopifnot("incorrect shape variable in answer5, specify a correct one if required"= setequal(digest(paste(toString(rlang::get_expr(c(answer5$layers[[1]]$mapping, answer5$mapping)$shape)), "e6096")), "b08e17dd660603103e2ee28ab64c28dc"))
stopifnot("the colour label in answer5 is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer5$layers[[1]]$mapping, answer5$mapping)$colour) != answer5$labels$colour), "e6096")), "746d1a67b388ecdb3b41e2991f6c7a9b"))
stopifnot("the shape label in answer5 is not descriptive, nicely formatted, or human readable"= setequal(digest(paste(toString(rlang::get_expr(c(answer5$layers[[1]]$mapping, answer5$mapping)$colour) != answer5$labels$shape), "e6096")), "b08e17dd660603103e2ee28ab64c28dc"))
stopifnot("fill variable in answer5 is not correct"= setequal(digest(paste(toString(quo_name(answer5$mapping$fill)), "e6096")), "49a18b64e26ec5d8583be9c0f212390e"))
stopifnot("fill label in answer5 is not informative"= setequal(digest(paste(toString((quo_name(answer5$mapping$fill) != answer5$labels$fill)), "e6096")), "b08e17dd660603103e2ee28ab64c28dc"))
stopifnot("position argument in answer5 is not correct"= setequal(digest(paste(toString(class(answer5$layers[[1]]$position)[1]), "e6096")), "c2a4e013a5e47f08cdbd8ef38edd19f9"))

print('Success!')

### PART 2: Web Scraping

Web scraping involves automatically extracting data from websites. Unlike APIs, which provide structured access to data, scraping involves pulling data directly from HTML content. Web scraping is commonly used to extract data from online sources when there are no API available to facilitate data acquisition. It is very important that before scraping any website we check its terms of service or robots.txt file to make sure web scraping is allowed for that webpage.

For this part we will be using the `rvest` package in R to extract data from HTML. Using `rvest` for web scraping requires basic understanding of HTML including the structure of a HTML content (elements, attributes). `rvest` functions makes it easier to access the required data from a HTML content and some of the commonly used functions include: 

- `read_html()`: Loads HTML content from a webpage URL or a character string.  

- `html_element()`: Retrieve a single matching element based on a CSS selector.  

- `html_elements()`: Retrieves all matching elements using CSS selectors.  

- `html_table()`: Converts HTML tables into data frames.  

- `html_text()/html_text2()`: Gets the text content from within HTML tags.  

- `html_name()`: Returns the tag name(s) of HTML elements.  

- `html_attr()`: Retrieves a single attribute.  

- `html_attrs()`: Retrieves all attributes.  

Let's get back to the Frankfurter exchange rate data. Suppose you don’t have access to the API, but someone has embedded a historical CAD exchange rate table on a webpage. Instead of downloading a CSV, we want to extract the table directly using web scraping.

In [ ]:
url <-"https://ubc-dsci.github.io/dsci-200/schedule/data/currency-data.html"
page <- read_html(url)

In [ ]:
page

**Question 6**

Using the CSS selectors, scrape only the headers of the table from the HTML content. HINT: You can use the SelectorGadget Chrome extension to identify the nodes and then use `html_nodes()` to extract the elements. These extracted nodes needs to be further processed using a parser function like `html_text()/html_text2()` to make the output readable. Assign the parsed output to variable `answer6`. (1 point)

In [ ]:
# answer6 <- "FILL IN YOUR ANSWER HERE"

# YOUR CODE HERE
fail()

answer6

In [ ]:
library(digest)
stopifnot("type of answer6 is not character"= setequal(digest(paste(toString(class(answer6)), "49b0c")), "c80cddb82b76bfee1b00f590829c6bc2"))
stopifnot("length of answer6 is not correct"= setequal(digest(paste(toString(length(answer6)), "49b0c")), "04698123cb136757a63e6765a0472f39"))
stopifnot("value of answer6 is not correct"= setequal(digest(paste(toString(tolower(answer6)), "49b0c")), "30f584720e254ed59500131faf2852db"))
stopifnot("letters in string value of answer6 are correct but case is not correct"= setequal(digest(paste(toString(answer6), "49b0c")), "34d633a3963bd27bc11f870ef6a11b1e"))

print('Success!')

**Question 7**

Using the CSS selectors, scrape only the `date` column from the table and parse it to a readable format. Assign the parsed output to variable `answer7`. HINT: Is there a CSS selector that can select the first cell in any table row? (1 point)

In [ ]:
# answer7 <- "FILL IN YOUR ANSWER HERE"

# YOUR CODE HERE
fail()

head(answer7)

In [ ]:
library(digest)
stopifnot("type of answer7 is not character"= setequal(digest(paste(toString(class(answer7)), "ac663")), "ad1d5f7103da89520f60a8cda4f35f2c"))
stopifnot("length of answer7 is not correct"= setequal(digest(paste(toString(length(answer7)), "ac663")), "8a20600408a4a00f52de68ff4ff3ac51"))
stopifnot("value of answer7 is not correct"= setequal(digest(paste(toString(tolower(answer7)), "ac663")), "04009933f1c3b5debe8dd685556b395f"))
stopifnot("letters in string value of answer7 are correct but case is not correct"= setequal(digest(paste(toString(answer7), "ac663")), "04009933f1c3b5debe8dd685556b395f"))

print('Success!')

**Question 8**

Extract the entire table from the HTML content using html_table() as a tibble. Assign the tibble to variable `answer8`. HINT: Use `pluck()` to extract the tibble from `html_table()`. (1 point)

In [ ]:
# answer8 <- "FILL IN YOUR ANSWER HERE"

# YOUR CODE HERE
fail()

head(answer8)

In [ ]:
library(digest)
stopifnot("answer8 should be a data frame"= setequal(digest(paste(toString('data.frame' %in% class(answer8)), "b2b35")), "f28006be921c81354c08b60eb39e65b0"))
stopifnot("dimensions of answer8 are not correct"= setequal(digest(paste(toString(dim(answer8)), "b2b35")), "e844da4a1b1141eaed185badbda66552"))
stopifnot("column names of answer8 are not correct"= setequal(digest(paste(toString(sort(colnames(answer8))), "b2b35")), "e066a9c92d6595cd285a2536b8d52f8c"))
stopifnot("types of columns in answer8 are not correct"= setequal(digest(paste(toString(sort(unlist(sapply(answer8, class)))), "b2b35")), "4b746eecfc17a6f032ac56ae9bc607c0"))
stopifnot("values in one or more numerical columns in answer8 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer8, is.numeric))) sort(round(sapply(answer8[, sapply(answer8, is.numeric)], sum, na.rm = TRUE), 2)) else 0), "b2b35")), "b5d5106cbda45034b7b9a73a20c6ec55"))
stopifnot("values in one or more character columns in answer8 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer8, is.character))) sum(sapply(answer8[sapply(answer8, is.character)], function(x) length(unique(x)))) else 0), "b2b35")), "c4a2a4a3179f34194f444b12a9aae651"))
stopifnot("values in one or more factor columns in answer8 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer8, is.factor))) sum(sapply(answer8[, sapply(answer8, is.factor)], function(col) length(unique(col)))) else 0), "b2b35")), "c3af8ca227fa54baffd57d4e293b97b7"))

print('Success!')

### PART 3: Parquet

Apache Parquet is a free, open-source file format designed to store and process large amounts of data efficiently. It organizes data by column instead of by row, which can make storage smaller and speed up analysis. Parquet is useful when you’re working with complex structures like lists and nested records or large datasets, especially in situations where you’re only interested in a few columns at a time. Since the format stores data column by column, it can compress better and skip over unneeded data, reduing computational time needed. We will use the `arrow` package to work with Apache Parquet files in R. 

In this section we will try to write the data scraped in `answer8` to write and then read files in parquet format.

Let's create a new column with the year for each data points.

In [ ]:
cad_parquet_data <- answer8 |> 
  mutate(year = year(date))

cad_parquet_data |> group_by(year) |>  summarise(count = n())

**Question 9**

Replace the ... in the following code chunck to write `cad_parquet_data` into a a set of parquet files based on the year. HINT: use `group_by` and `write_dataset`. (1 point)

In [ ]:
pq_path <- "./pq_files/"

if (!dir.exists(pq_path)) {
  dir.create(pq_path)
}

# cad_parquet_data |> 
#     ...(year) |> 
#     ...(..., format = ...)

# answer9 <- list.files(..., full.names = TRUE)


# YOUR CODE HERE
fail()


In [ ]:
library(digest)
stopifnot("type of answer9 is not character"= setequal(digest(paste(toString(class(answer9)), "76b1e")), "4db85a1d795e6ec0ba9f03047b952a2f"))
stopifnot("length of answer9 is not correct"= setequal(digest(paste(toString(length(answer9)), "76b1e")), "646c96b09ba5f17cc1b0118c2041b212"))
stopifnot("value of answer9 is not correct"= setequal(digest(paste(toString(tolower(answer9)), "76b1e")), "2813131000e62c9bc69b9f5557cc9518"))
stopifnot("letters in string value of answer9 are correct but case is not correct"= setequal(digest(paste(toString(answer9), "76b1e")), "2813131000e62c9bc69b9f5557cc9518"))

print('Success!')

**Question 10**

Read the written parquet files and expand the files. Store the expanded tibble in `answer10`. (1 point) 

In [ ]:
#answer10<-"FILL IN YOUR ANSWER HERE"

# YOUR CODE HERE
fail()

head(answer10)

In [ ]:
library(digest)
stopifnot("answer10 should be a data frame"= setequal(digest(paste(toString('data.frame' %in% class(answer10)), "30d9")), "429c29390005b844a64a91e92e0e1db8"))
stopifnot("dimensions of answer10 are not correct"= setequal(digest(paste(toString(dim(answer10)), "30d9")), "2b695b0439ec7a5723ed4317d817497f"))
stopifnot("column names of answer10 are not correct"= setequal(digest(paste(toString(sort(colnames(answer10))), "30d9")), "4557fe0acc02a03197812610b5609c76"))
stopifnot("types of columns in answer10 are not correct"= setequal(digest(paste(toString(sort(unlist(sapply(answer10, class)))), "30d9")), "f04ef356bfa7f06abd8285f7f70e7961"))
stopifnot("values in one or more numerical columns in answer10 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer10, is.numeric))) sort(round(sapply(answer10[, sapply(answer10, is.numeric)], sum, na.rm = TRUE), 2)) else 0), "30d9")), "f4c1b5656f35f88fb675c14cfd138025"))
stopifnot("values in one or more character columns in answer10 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer10, is.character))) sum(sapply(answer10[sapply(answer10, is.character)], function(x) length(unique(x)))) else 0), "30d9")), "fcd951a15fccac6686e581b32260c5e5"))
stopifnot("values in one or more factor columns in answer10 are not correct"= setequal(digest(paste(toString(if (any(sapply(answer10, is.factor))) sum(sapply(answer10[, sapply(answer10, is.factor)], function(col) length(unique(col)))) else 0), "30d9")), "5e5ed1d3a73e29ac9d4b40f00068d980"))

print('Success!')